In [ ]:
# Data : 
#     +) https://www.kaggle.com/datasets/aibloy/fairface
#     +) https://www.kaggle.com/datasets/jangedoo/utkface-new

In [ ]:
import os
import cv2
import math
import json
import random
import glob
import subprocess
from io import BytesIO
from typing import Dict
from collections import Counter, OrderedDict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.serialization
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)
from torchvision import models
from torchvision import transforms as T
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import (
    LinearLR,
    CosineAnnealingLR,
    SequentialLR,
)
import requests
from tqdm.notebook import tqdm
from IPython.display import FileLink, display

# Data processing

In [ ]:
def set_seed(seed=42):
    """
    Thiết lập ngẫu nhiên (seed) cố định.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Kích thước ảnh đầu vào
IMAGE_SIZE = 112

# Chỉ số trung bình và độ lệch chuẩn của bộ ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Transform cho tập Train
train_tf = T.Compose([
    T.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.9, 1.0), # Zoom nhẹ ngẫu nhiên
        ratio=(0.95, 1.05) # Thay đổi tỉ lệ khung hình
    ),
    T.RandomHorizontalFlip(p=0.5), # Lật ngang ngẫu nhiên
    T.ColorJitter(                 # Thay đổi màu sắc nhẹ
        brightness=0.1,
        contrast=0.1,
        saturation=0.05,
        hue=0.02
    ),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Transform cho tập Val/Test
val_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

def load_fairface(base_dir: str):
    """
    Đọc và xử lý dữ liệu từ bộ FairFace.
    """
    if not os.path.exists(base_dir):
        print(f"Không tìm thấy thư mục: {base_dir}")
        return pd.DataFrame(), pd.DataFrame()

    train_df = pd.read_csv(os.path.join(base_dir, "train_labels.csv"))
    val_df   = pd.read_csv(os.path.join(base_dir, "val_labels.csv"))

    # Mapping dữ liệu dạng chuỗi sang số
    gender_map = {'Male': 0, 'Female': 1}
    race_map = {
        'White': 0,
        'Black': 1,
        'Latino_Hispanic': 2,
        'East Asian': 3,
        'Southeast Asian': 3, # Gộp chung Đông Á và ĐNA thành nhóm 3
        'Indian': 4,
        'Middle Eastern': None # Bỏ qua nhóm này
    }

    # Gom nhóm tuổi (Age Binning)
    # FairFace gốc chia 9 nhóm, gom thành 6 nhóm lớn
    age_map_6 = {
        '0-2': 0, '3-9': 0,        # 0–9 tuổi -> Nhóm 0
        '10-19': 1,                # 10–19 tuổi -> Nhóm 1
        '20-29': 2,                # 20–29 tuổi -> Nhóm 2
        '30-39': 3,                # 30–39 tuổi -> Nhóm 3
        '40-49': 4, '50-59': 4,    # 40–59 tuổi -> Nhóm 4
        '60-69': 5, 'more than 70': 5  # 60+ tuổi -> Nhóm 5
    }

    for df, subset in [(train_df, "train"), (val_df, "val")]:
        # Đường dẫn ảnh
        df["file"] = df["file"].apply(
            lambda f: os.path.join(base_dir, f.replace(f"{subset}/{subset}", subset))
        )

        # Áp dụng mapping
        df["gender"] = df["gender"].map(gender_map)
        df["race"]   = df["race"].map(race_map)
        df["age"]    = df["age"].map(age_map_6)

        # Bỏ các dòng thiếu dữ liệu
        df.dropna(subset=["gender", "race", "age"], inplace=True)
        df["race"] = df["race"].astype(int)

    cols = ["file", "age", "race", "gender"]
    return train_df[cols], val_df[cols]

def load_utkface(utk_dir: str):
    """
    Đọc dữ liệu từ bộ UTKFace.
    Tên file chứa thông tin: [Age]_[Gender]_[Race]_[Date].jpg
    """
    if not os.path.exists(utk_dir):
        print(f"Không tìm thấy thư mục: {utk_dir}")
        return pd.DataFrame(), pd.DataFrame()

    recs = []
    # Quét file và parse tên file
    for f in os.listdir(utk_dir):
        if not f.lower().endswith((".jpg", ".png", ".jpeg")): continue

        parts = f.split("_")
        if len(parts) < 4: continue

        try:
            age, gender, race = int(parts[0]), int(parts[1]), int(parts[2])
        except ValueError: continue

        if age > 120: continue 

        recs.append([os.path.join(utk_dir, f), age, gender, race])

    df = pd.DataFrame(recs, columns=["file", "age_raw", "gender", "race"])

    # Hàm chuyển tuổi sang 6 nhóm
    def age_to_bin6(age):
        if age <= 9: return 0
        if age <= 19: return 1
        if age <= 29: return 2
        if age <= 39: return 3
        if age <= 59: return 4
        return 5 # 60+

    df["age"] = df["age_raw"].apply(age_to_bin6)

    # Mapping Race của UTK
    race_mapping = {
        0: 0,   # White
        1: 1,   # Black
        2: 3,   # Asian -> Map vào nhóm 3 (Asian)
        3: 4,   # Indian -> Map vào nhóm 4 (Indian)
        4: None # Others -> Bỏ qua
    }

    df["race"] = df["race"].map(race_mapping)
    df.dropna(subset=["race"], inplace=True)
    df["race"] = df["race"].astype(int)

    df = df[["file", "age", "race", "gender"]]

    # Chia train/val tỉ lệ 80/20, giữ nguyên tỉ lệ Race (stratify)
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df["race"]
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

class MultiTaskFaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["file"]

        try:
            # Mở ảnh và convert RGB
            img = Image.open(image_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (112, 112), (0, 0, 0))

        if self.transform:
            img = self.transform(img)

        # Trả về ảnh và dictionary chứa 3 nhãn
        return img, {
            "age": torch.tensor(row["age"], dtype=torch.long),
            "gender": torch.tensor(row["gender"], dtype=torch.long),
            "race": torch.tensor(row["race"], dtype=torch.long),
        }

def fast_check_leakage(df1, df2, name1="Set1", name2="Set2"):
    """
    Kiểm tra file ảnh nào bị trùng lặp giữa 2 tập dữ liệu (Data Leakage).
    """
    s1 = set(df1['file'])
    s2 = set(df2['file'])
    overlap = len(s1.intersection(s2))
    
    if overlap > 0:
        print(f"LEAKAGE {name1}-{name2}: Có {overlap} ảnh trùng lặp!")
    else:
        print(f"{name1}-{name2}: Không có trùng lặp.")


set_seed(42)
FAIRFACE_DIR = "/kaggle/input/fairface/FairFace"
UTK_DIR = "/kaggle/input/utkface-new/UTKFace"

# Load data
ff_train, ff_val = load_fairface(FAIRFACE_DIR)
utk_train, utk_val = load_utkface(UTK_DIR) 

# Gộp dữ liệu
# Train = FF Train + UTK Train
train_df = pd.concat([ff_train, utk_train], ignore_index=True)
# Temp Val = FF Val + UTK Val
temp_val = pd.concat([ff_val, utk_val], ignore_index=True)

# Tách tập TEST từ tập Val (50% Val, 50% Test)
if len(temp_val) > 0:
    val_df, test_df = train_test_split(
        temp_val, 
        test_size=0.2, 
        random_state=42, 
        stratify=temp_val["race"]
    )
else:
    val_df, test_df = pd.DataFrame(), pd.DataFrame()

# Reset index
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Kiểm tra leakage dữ liệu
fast_check_leakage(train_df, val_df, "Train", "Val")
fast_check_leakage(train_df, test_df, "Train", "Test")
fast_check_leakage(val_df, test_df, "Val", "Test")


print(f"TRAIN: {len(train_df):,} ảnh")
print(f"VAL:   {len(val_df):,} ảnh")
print(f"TEST:  {len(test_df):,} ảnh")

print("\nPhân bố chủng tộc trong tập Test:")
print(test_df['race'].value_counts().sort_index().to_dict())

# Dataset & DataLoader
train_ds = MultiTaskFaceDataset(train_df, train_tf)
val_ds   = MultiTaskFaceDataset(val_df, val_tf)
test_ds  = MultiTaskFaceDataset(test_df, val_tf) 

train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
    prefetch_factor=4
)

val_loader = DataLoader(
    val_ds,
    batch_size=256,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
# Nhãn nhóm tuổi (Age Bins)
AGE_LABELS = {
    0: "0–9",   # Trẻ em
    1: "10–19", # Thiếu niên
    2: "20–29", # Thanh niên
    3: "30–39", # Trung niên sớm
    4: "40–59", # Trung niên
    5: "60+",   # Người cao tuổi
}

# Nhãn chủng tộc (Race Groups)
RACE_LABELS = {
    0: "White",   # Da trắng
    1: "Black",   # Da đen
    2: "Latino",  # Gốc Latin/Hispanic
    3: "Asian",   # Châu Á (Đông Á + Đông Nam Á)
    4: "Indian"   # Ấn Độ
}

# Nhãn giới tính
GENDER_LABELS = {
    0: "Male",   # Nam
    1: "Female"  # Nữ
}

def plot_label_distributions(df, name="Train"):
    """
    Vẽ 3 biểu đồ cột (Bar Chart) hiển thị sự phân bố của Age, Race, Gender trong DataFrame.
    
    Args:
        df (pd.DataFrame): DataFrame chứa dữ liệu (cần có cột 'age', 'race', 'gender').
        name (str): Tên tập dữ liệu (VD: 'Train', 'Val', 'Test') hiển thị tiêu đề.
    """
    # Tạo khung hình chứa 3 biểu đồ con nằm ngang (1 hàng, 3 cột)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Dictionary ánh xạ tên cột sang dictionary nhãn tương ứng
    label_maps = {
        "age": AGE_LABELS,
        "race": RACE_LABELS,
        "gender": GENDER_LABELS
    }

    # Duyệt qua từng thuộc tính
    for i, col in enumerate(["age", "race", "gender"]):
        # Đếm số lượng mẫu cho từng giá trị và sắp xếp theo index (0, 1, 2...)
        counts = df[col].value_counts().sort_index()

        # Biểu đồ cột
        axes[i].bar(counts.index, counts.values, color='steelblue', edgecolor='black', alpha=0.7)
        axes[i].set_title(f"{name} {col.capitalize()} Distribution", fontweight="bold")
        axes[i].set_ylabel("Count")

        # Thiết lập nhãn trục X (thay số 0,1,2 bằng text 'White', 'Black'...)
        axes[i].set_xticks(counts.index)
        axes[i].set_xticklabels(
            [label_maps[col][k] for k in counts.index],
            rotation=30 if col == "age" else 0 
        )

    # Tự động căn chỉnh khoảng cách giữa các biểu đồ
    plt.tight_layout()
    plt.show()

plot_label_distributions(train_df, "Train")
plot_label_distributions(val_df, "Validation")
plot_label_distributions(test_df, "Test")

In [ ]:
def visualize_augmentations(dataset, num_samples=5):
    """
    Hàm trực quan hóa so sánh ảnh gốc và ảnh sau khi Augmentation.
    """
    # Lấy ngẫu nhiên index
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 3 * num_samples))
    plt.subplots_adjust(wspace=0.2, hspace=0.2)
    
    # Tiêu đề cột
    axes[0, 0].set_title("Original (Raw Image)\n", fontsize=14, fontweight='bold')
    axes[0, 1].set_title("Transformed (Model Input)\n", fontsize=14, fontweight='bold')

    for i, idx in enumerate(indices):
        # Lấy dữ liệu từ Dataset row lấy từ dataframe gốc để load ảnh thô
        row = dataset.df.iloc[idx]
        img_path = row['file']
        label_str = row['gender']
        
        # Ảnh gốc (Raw)
        original_img = Image.open(img_path).convert('RGB')
        
        # Ảnh đã biến đổi (Transformed)
        transformed_tensor, label_idx = dataset[idx]
        
        # Xử lý Tensor để hiển thị (Denormalize)
        img_display = transformed_tensor.numpy().transpose(1, 2, 0) # [1, 112, 112] -> [112, 112, 1]
        img_display = (img_display * 0.5) + 0.5 # Denormalize: x * std + mean
        img_display = np.clip(img_display, 0, 1) # Giá trị không vượt quá giới hạn
        
        # Cột 1: Ảnh gốc
        axes[i, 0].imshow(original_img)
        axes[i, 0].axis('off')
        axes[i, 0].text(0, -5, f"Label: {label_str}", color='blue', fontsize=10)
        
        # Cột 2: Ảnh sau khi Augment
        # cmap='gray' vì ảnh đầu ra là ảnh xám
        axes[i, 1].imshow(img_display.squeeze(), cmap='gray') 
        axes[i, 1].axis('off')
        axes[i, 1].text(0, -5, f"Class ID: {label_idx['age']}", color='red', fontsize=10)

    plt.show()

visualize_augmentations(train_ds, num_samples=6)

# Model structure

In [ ]:
def age_to_ordinal_soft(targets, num_classes, epsilon=0.1):
    """
    Chuyển nhãn tuổi sang dạng Soft Ordinal (bài toán hồi quy thứ tự).
    
    - Ordinal: Nếu 30 tuổi, thì cũng >10, >20 tuổi. Vector: [1, 1, 1, 0, 0]
    - Soft: Vector: [0.95, 0.95, 0.95, 0.05, 0.05]
    """
    # Tạo vector thứ tự [0, 1, 2, ..., K-2]
    rank = torch.arange(num_classes - 1, device=targets.device)
    
    # Mở rộng chiều để so sánh (Broadcasting)
    targets = targets.unsqueeze(1).float()
    rank = rank.unsqueeze(0).float()
    
    # Tạo nhãn cứng (Hard Label): 1 nếu tuổi > rank, 0 nếu ngược lại
    hard_labels = (targets > rank).float()
    
    # Làm mềm nhãn (Soft Label)
    soft_labels = hard_labels * (1 - epsilon) + (epsilon / 2)
    return soft_labels

class SpatialAttention(nn.Module):
    """
    Cơ chế chú ý không gian (Spatial Attention Module - SAM).
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        # Input channel = 2
        # Output channel = 1
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid() # Đưa về khoảng [0, 1]

    def forward(self, x):
        # x shape: [Batch, Channel, H, W]
        
        avg_out = torch.mean(x, dim=1, keepdim=True) # [B, 1, H, W]
        max_out, _ = torch.max(x, dim=1, keepdim=True) # [B, 1, H, W]
        
        # Ghép lại
        x_cat = torch.cat([avg_out, max_out], dim=1) # [B, 2, H, W]
        
        attn = self.sigmoid(self.conv(x_cat)) # [B, 1, H, W]
        
        # Nhân trọng số vào Feature Map gốc
        return x * attn


class MultiTaskEfficientNetB0(nn.Module):
    def __init__(
        self,
        num_age_classes=6,
        dropout=0.5,
        freeze_backbone=True,
        gender_weight=None, 
        race_weight=None,
    ):
        super().__init__()
        self.num_age_classes = num_age_classes

        # Tải weights ImageNet
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        backbone = efficientnet_b0(weights=weights)
        self.features = backbone.features

        # EfficientNet gốc giảm size ảnh đi 1 nửa layer đầu (Stride=2).
        # Với ảnh input nhỏ (112x112), cần giữ size để không mất chi tiết.
        
        # Lấy layer conv đầu tiên
        old_conv = self.features[0][0]
        
        # Tạo layer giống nhưng với Stride=1
        new_conv = nn.Conv2d(
            in_channels=old_conv.in_channels,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=1, 
            padding=old_conv.padding,
            bias=False
        )
        
        # Copy trọng số đã học từ ImageNet.
        with torch.no_grad():
            new_conv.weight.copy_(old_conv.weight)
            
        # Gán lại vào mạng
        self.features[0][0] = new_conv

        self.spatial_attention = SpatialAttention(kernel_size=7)
        self.pool = nn.AdaptiveAvgPool2d(1)

        # FREEZE STRATEGY
        if freeze_backbone:
            # Đóng băng toàn bộ backbone
            for p in self.features.parameters():
                p.requires_grad = False
            
            # Mở Layer đầu (do vừa đổi stride -> học lại)
            for p in self.features[0].parameters():
                p.requires_grad = True
            
            # Mở băng các Block cuối (học đặc trưng khuôn mặt)
            for name, p in self.features.named_parameters():
                if "features.6." in name or "features.7." in name:
                    p.requires_grad = True

        # Kích thước vector đặc trưng của EfficientNet-B0
        feat_dim = 1280

        # Gender: Phân loại 2 lớp
        self.gender_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 2))
        
        # Race: Phân loại 5 lớp
        self.race_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 5))
        
        # Age: Ordinal Regression (6 lớp)
        self.age_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_age_classes - 1) 
        )

        # LOSS FUNCTIONS
        self.loss_gender_fn = nn.CrossEntropyLoss(weight=gender_weight, label_smoothing=0.1)
        self.loss_race_fn = nn.CrossEntropyLoss(weight=race_weight, label_smoothing=0.1)
        self.loss_age_fn = nn.BCEWithLogitsLoss()

        # UNCERTAINTY WEIGHTING
        # Chỉnh hệ số (vd: 0.5*Gender + 1.5*Age...) -> tự học.
        # Task nhiễu cao (khó học) -> Variance cao -> Trọng số Loss giảm.
        self.log_vars = nn.Parameter(torch.zeros((3)))

    def forward(self, x):
        # Trích xuất đặc trưng từ Backbone
        x = self.features(x) # [B, 1280, H, W]

        # Tập trung vào vùng mặt, bỏ qua nền nhiễu
        x = self.spatial_attention(x) 
        
        # Global Average Pooling & Flatten
        x = self.pool(x).flatten(1) # [B, 1280]

        return {
            "gender": self.gender_head(x),
            "race":   self.race_head(x),
            "age":    self.age_head(x)
        }
    
    def compute_loss(self, outputs, targets):
        """Tính tổng Loss sử dụng Uncertainty Weights"""
        loss_g = self.loss_gender_fn(outputs["gender"], targets["gender"])
        loss_r = self.loss_race_fn(outputs["race"], targets["race"])
        
        # Chuyển nhãn tuổi sang vector Ordinal Soft
        ordinal_labels = age_to_ordinal_soft(targets["age"], self.num_age_classes, epsilon=0.1)
        loss_a = self.loss_age_fn(outputs["age"], ordinal_labels)

        # Loss Uncertainty (Kendall et al.):
        # Loss = exp(-log_var) * Task_Loss + log_var
        # log_var đóng vai trò như hệ số phạt (regularization)
        loss_total_g = torch.exp(-self.log_vars[0]) * loss_g + self.log_vars[0]
        loss_total_r = torch.exp(-self.log_vars[1]) * loss_r + self.log_vars[1]
        loss_total_a = torch.exp(-self.log_vars[2]) * loss_a + self.log_vars[2]

        return loss_total_g + loss_total_r + loss_total_a

    @torch.no_grad()
    def predict(self, x):
        """Hàm dự đoán (Inference)"""
        self.eval()
        out = self(x)
        
        # Decode tuổi từ Ordinal Output
        # Chuyển logits thành xác suất [0-1]
        age_probs = torch.sigmoid(out["age"])
        # Đếm số lượng đầu ra vượt qua ngưỡng 0.5
        # Ví dụ: [0.9, 0.8, 0.2, 0.1, 0.05] -> >0.5 là [1, 1, 0, 0, 0] -> Sum = 2
        # => Dự đoán thuộc nhóm tuổi số 2 (20-29 tuổi)
        pred_age = (age_probs > 0.5).sum(dim=1)
        
        return {
            "age": pred_age,
            # Chọn class có xác suất cao nhất (argmax)
            "gender": torch.argmax(out["gender"], dim=1),
            "race": torch.argmax(out["race"], dim=1)
        }

# Train - Evaluate

In [ ]:
def calculate_weights(df, column, device):
    """Tính class weight cho bài toán phân loại (Gender, Race)"""
    labels = df[column].to_numpy()
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    return torch.tensor(weights, dtype=torch.float32).to(device)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

# GradScaler hỗ trợ Mixed Precision Training (FP16)
scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)

torch.serialization.add_safe_globals([
    np.core.multiarray.scalar, np.dtype,
    np.int64, np.int32, np.int16, np.int8,
    np.uint8, np.float32, np.float64
])

def safe_load(path):
    return torch.load(path, map_location=device, weights_only=False)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

def run_epoch(model, loader, optimizer, scheduler, scaler, phase="train"):

    is_train = phase == "train"

    core = model.module if isinstance(model, nn.DataParallel) else model
    
    core.train() if is_train else core.eval()

    total_loss = 0.0
    total = 0
    
    # Khởi tạo các biến đếm
    correct_gender = 0
    correct_race = 0
    correct_age = 0
    age_pm1_correct = 0 # Độ chính xác +-1 tuổi (chấp nhận sai số 1 đơn vị)
    age_mae_sum = 0.0   # Mean Absolute Error của tuổi

    # Thanh tiến trình
    pbar = tqdm(loader, desc=f"{phase.capitalize():<5}", leave=False)

    for imgs, labels in pbar:
        imgs = imgs.to(device, non_blocking=True)
        l_age = labels["age"].to(device, non_blocking=True)
        l_gender = labels["gender"].to(device, non_blocking=True)
        l_race = labels["race"].to(device, non_blocking=True)

        targets = {"age": l_age, "gender": l_gender, "race": l_race}

        # FORWARD PASS
        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=use_cuda):
                outputs = core(imgs)
                loss = core.compute_loss(outputs, targets)

        # BACKWARD PASS
        if is_train:
            optimizer.zero_grad(set_to_none=True) # Reset gradient
            scaler.scale(loss).backward()         # Scale loss
            scaler.unscale_(optimizer)            # Unscale
            torch.nn.utils.clip_grad_norm_(core.parameters(), 5.0) # Clip gradient
            
            scaler.step(optimizer)                # Update weights
            scaler.update()                       # Update scaler factor

            if scheduler is not None:
                scheduler.step()                  # Update Learning Rate

        # METRICS
        bs = imgs.size(0)
        total += bs
        total_loss += loss.item() * bs

        with torch.no_grad():
            # Tính accuracy cho Gender & Race
            correct_gender += (outputs["gender"].argmax(1) == l_gender).sum().item()
            correct_race += (outputs["race"].argmax(1) == l_race).sum().item()
            
            # Tính metrics cho Age (Ordinal Regression)
            # Output là logits -> Sigmoid -> Xác suất > 0.5 là 1, ngược lại 0
            # Tổng số lượng số 1 là nhãn tuổi dự đoán
            # [1, 1, 1, 0, 0] -> sum = 3 -> Tuổi dự đoán là 3
            probs = torch.sigmoid(outputs["age"])
            age_pred = (probs > 0.5).sum(dim=1)

            correct_age += (age_pred == l_age).sum().item()
            # Age +- 1 Accuracy: Chấp nhận sai số 1 nhóm tuổi
            age_pm1_correct += (torch.abs(age_pred - l_age) <= 1).sum().item()
            # Age MAE: Sai số tuyệt đối trung bình
            age_mae_sum += torch.abs(age_pred.float() - l_age.float()).sum().item()

        # Cập nhật pbar
        if is_train:
            pbar.set_postfix({
                "loss": f"{total_loss / total:.4f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
            })

    # Kết quả
    metrics = {
        "loss": total_loss / total,
        "gender_acc": correct_gender / total,
        "race_acc": correct_race / total,
        "age_acc": correct_age / total,
        "age_mae": age_mae_sum / total,
        "age_pm1_acc": age_pm1_correct / total,
    }
    
    # Chỉ số trung bình
    metrics["avg_acc"] = (metrics["gender_acc"] + metrics["race_acc"] + metrics["age_acc"]) / 3

    return metrics

def build_optimizer_and_scheduler(model, loader, num_epochs, phase):
    """
    Tạo Optimizer với Learning Rate.
    """
    core = model.module if isinstance(model, nn.DataParallel) else model

    # Phân loại tham số: Backbone vs New Params (Heads, Attention...)
    backbone_params, new_params = [], []
    for n, p in core.named_parameters():
        if "features" in n: # backbone
            backbone_params.append(p)
        else:
            new_params.append(p)

    if phase == "freeze":
        # Giai đoạn 1: Đóng băng Backbone, train các lớp mới
        for p in backbone_params:
            p.requires_grad = False
        
        # LR cao (3e-3) lớp mới
        optimizer = torch.optim.AdamW([
            {"params": new_params, "lr": 3e-3}
        ], weight_decay=1e-4)
        
        max_lr = 3e-3
        
    else: # phase == "unfreeze"
        # Giai đoạn 2: Mở băng toàn bộ, fine-tune all parameters
        for p in backbone_params:
            p.requires_grad = True
            
        optimizer = torch.optim.AdamW([
            {"params": backbone_params, "lr": 1e-4},
            {"params": new_params, "lr": 1e-3},
        ], weight_decay=1e-4)
        
        max_lr = [1e-4, 1e-3]

    # OneCycleLR: Tăng LR từ từ lên max rồi giảm dần
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=num_epochs,
        steps_per_epoch=len(loader)
    )
    return optimizer, scheduler

def train_model(model, train_loader, val_loader, num_epochs=20, unfreeze_epoch=5, save_dir="./checkpoints", checkpoint_path=None):
    os.makedirs(save_dir, exist_ok=True)
    set_seed(42)
    
    # Setup Model & Multi-GPU
    model = model.to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
        model = nn.DataParallel(model)

    # Load checkpoint
    start_epoch = 0
    best_val_loss = float("inf")
    phase = "freeze"
    history = {"train": [], "val": []}

    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"Resuming from {checkpoint_path}...")
        ckpt = safe_load(checkpoint_path)
        
        # Load weights
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(ckpt["model_state_dict"])
        else:
            model.load_state_dict(ckpt["model_state_dict"])
            
        start_epoch = ckpt["epoch"] + 1
        best_val_loss = ckpt["best_val_loss"]
        phase = ckpt["phase"]
        if "history" in ckpt: history = ckpt["history"]

    # Khởi tạo Optimizer
    optimizer, scheduler = build_optimizer_and_scheduler(
        model, train_loader, num_epochs - start_epoch, phase
    )

    # Vòng lặp Epoch
    for epoch in range(start_epoch, num_epochs):
        print(f"\nEPOCH {epoch}/{num_epochs} | Phase: {phase}")

        # Unfreeze
        if epoch == unfreeze_epoch and phase == "freeze":
            phase = "unfreeze"
            # Re-build optimizer
            optimizer, scheduler = build_optimizer_and_scheduler(
                model, train_loader, num_epochs - epoch, phase
            )

        # Run Train & Val
        train_metrics = run_epoch(model, train_loader, optimizer, scheduler, scaler, "train")
        val_metrics   = run_epoch(model, val_loader, None, None, scaler, "val")
        
        # Logging
        print(f"Train Loss: {train_metrics['loss']:.4f} | Gender_Acc: {train_metrics['gender_acc']:.4f} | Race_Acc: {train_metrics['race_acc']:.4f} | Age_Acc: {train_metrics['age_acc']:.4f} | Avg_Acc: {train_metrics['avg_acc']:.4f}")
        print(f"Val Loss:   {val_metrics['loss']:.4f} | Gender_Acc: {val_metrics['gender_acc']:.4f} | Race_Acc: {val_metrics['race_acc']:.4f} | Age_Acc: {val_metrics['age_acc']:.4f} | AgeMAE: {val_metrics['age_mae']:.2f} | Avg_Acc: {val_metrics['avg_acc']:.4f}")

        history["train"].append({"epoch": epoch, **train_metrics})
        history["val"].append({"epoch": epoch, **val_metrics})

        # Save history (Json)
        with open(os.path.join(save_dir, "history.json"), "w") as f:
            json.dump(history, f, indent=2)

        # Save checkpoint
        save_dict = {
            "epoch": epoch,
            "phase": phase,
            "model_state_dict": model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_val_loss": best_val_loss,
            "history": history,
        }

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(save_dict, f"{save_dir}/best_model.pt")
            print("--> Saved BEST MODEL")

        torch.save(save_dict, f"{save_dir}/epoch_{epoch+1}.pt")

    print("\nTraining finished.")

In [ ]:
if __name__ == "__main__":
    race_weight = calculate_weights(train_df, "race", device)
    gender_weight = calculate_weights(train_df, "gender", device)

    model = MultiTaskEfficientNetB0(
        num_age_classes=6,
        dropout=0.3,          
        freeze_backbone=True,
        gender_weight=gender_weight,
        race_weight=race_weight,
    )

    train_model(
        model,
        train_loader,
        val_loader,
        num_epochs=40,
        checkpoint_path=""
    )

# Visualization & Test predict

In [ ]:
# Load dữ liệu
ckpt = safe_load("/kaggle/working/checkpoints/best_model.pt")
history = ckpt["history"]

# Trích xuất dữ liệu
train_loss = [x['loss'] for x in history['train']]
val_loss   = [x['loss'] for x in history['val']]

# Accuracies
val_acc_age    = [x['age_acc'] for x in history['val']]
val_acc_gender = [x['gender_acc'] for x in history['val']]
val_acc_race   = [x['race_acc'] for x in history['val']]

# Age Specific Metrics (MAE & PM1)
train_mae = [x['age_mae'] for x in history['train']]
val_mae   = [x['age_mae'] for x in history['val']]

train_pm1 = [x['age_pm1_acc'] for x in history['train']]
val_pm1   = [x['age_pm1_acc'] for x in history['val']]

epochs_range = range(1, len(train_loss) + 1)

# Vẽ biểu đồ (2 hàng, 2 cột)
plt.figure(figsize=(16, 12))

# Chart 1: Loss
plt.subplot(2, 2, 1)
plt.plot(epochs_range, train_loss, 'b-o', label='Train Loss')
plt.plot(epochs_range, val_loss, 'r-o', label='Val Loss')
plt.title('Total Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Chart 2: Task Accuracies
plt.subplot(2, 2, 2)
plt.plot(epochs_range, val_acc_gender, label='Gender Acc')
plt.plot(epochs_range, val_acc_race, label='Race Acc')
plt.plot(epochs_range, val_acc_age, 'g-s', linewidth=2, label='Age Exact Acc')
plt.title('Validation Accuracy per Task')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Chart 3: Age Analysis
plt.subplot(2, 2, 3)
plt.plot(epochs_range, val_acc_age, 'g-o', label='Exact Acc (Đúng tuyệt đối)')
plt.plot(epochs_range, val_pm1, 'm-^', label='PM1 Acc (Chấp nhận sai số 1)')
plt.title('Age Prediction Quality (Validation)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Chart 4: Age Error (MAE)
plt.subplot(2, 2, 4)
plt.plot(epochs_range, train_mae, 'b--', label='Train MAE')
plt.plot(epochs_range, val_mae, 'r-o', label='Val MAE')
plt.title('Age Mean Absolute Error (MAE)')
plt.xlabel('Epochs')
plt.ylabel('Error (Class Distance)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
GENDER_CLASSES = ["Male", "Female"]

RACE_CLASSES = [
    "White",
    "Black",
    "Latino",
    "Asian",
    "Indian",
]


AGE_CLASSES = [
    "0-9", "10-19", "20-29", "30-39",
    "40-59", "60+",
]

def get_all_predictions_multitask(model, loader, device, task):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)

            # XỬ LÝ AGE (ORDINAL)
            if task == "age":
                logits = outputs["age"]            # (B, C-1)
                probs = torch.sigmoid(logits)      
                # Đếm số ngưỡng vượt qua 0.5 để ra nhãn tuổi (0, 1, 2...)
                preds = (probs > 0.5).sum(dim=1).cpu().numpy()
                
                # Lấy nhãn thật
                y_pred.extend(preds)
                y_true.extend(labels["age"].numpy())

            # XỬ LÝ GENDER VÀ RACE
            else:
                logits = outputs[task]
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                
                y_pred.extend(preds)
                y_true.extend(labels[task].numpy())

    return np.array(y_true), np.array(y_pred)



def plot_confusion_matrix(y_true, y_pred, class_names):
    # Tính toán các chỉ số
    acc = accuracy_score(y_true, y_pred)
    print(f"\n{'='*30}")
    print(f"Accuracy = {acc*100:.2f}%")
    print(f"{'='*30}")
    
    # Precision, Recall, F1-Score
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    
    # Tính Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Heatmap
    plt.figure(figsize=(10, 8))
    
    # fmt='d': hiển thị số nguyên
    # cmap='Blues': tông màu xanh
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, 
                yticklabels=class_names)
    
    plt.title('Confusion Matrix', fontsize=16)
    plt.ylabel('Nhãn thực tế (True)', fontsize=12)
    plt.xlabel('Dự đoán của Model (Predicted)', fontsize=12)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Normalized Confusion Matrix (Theo phần trăm)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='OrRd', 
                xticklabels=class_names, 
                yticklabels=class_names)
    
    plt.title('Normalized Confusion Matrix (%)', fontsize=16)
    plt.ylabel('Nhãn thực tế (True)', fontsize=12)
    plt.xlabel('Dự đoán của Model (Predicted)', fontsize=12)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
class MultiTaskGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer # Layer Conv cuối cùng
        self.activations = None          # Feature Maps (A)
        self.gradients = None            # Gradients (G)
        self.hook_handle = None          # Quản lý việc bật/tắt theo dõi

    def _fwd(self, m, i, o):
        """
        Forward Hook: Chạy mỗi khi dữ liệu đi qua target_layer.
        """
        # Lưu lại Feature Maps
        self.activations = o
    
        # Đăng ký hook bắt Gradient trên Tensor
        if o.requires_grad:
            o.register_hook(self._save_grad)

    def _save_grad(self, grad):
        """Backward Hook: Lưu lại đạo hàm (Gradient)"""
        self.gradients = grad

    def enable(self):
        """BẬT chế độ theo dõi"""
        self.hook_handle = self.target_layer.register_forward_hook(self._fwd)

    def disable(self):
        """TẮT chế độ theo dõi"""
        if self.hook_handle is not None:
            self.hook_handle.remove()
            self.hook_handle = None

    def generate(self, x, task, class_idx):
        """
        Tạo Heatmap cho một ảnh đầu vào.
        
        Args:
            x: Ảnh đầu vào (Tensor)
            task: Tên nhánh nhiệm vụ (vd: 'emotion', 'gender')
            class_idx: Index của nhãn muốn giải thích (vd: 3 - Happy)
        """
        # Reset trạng thái cũ
        self.activations = None
        self.gradients = None
        self.model.zero_grad() # Xóa gradient cũ

        # Forward Pass
        out = self.model(x)
        logits = out[task] # Lấy kết quả task

        # Backward Pass
        # Tính gradient cho class mục tiêu (class_idx)
        score = logits[0, class_idx]
        
        # retain_graph=True
        score.backward(retain_graph=True)

        # Kiểm tra nếu không bắt được gradient (thường do layer bị freeze hoặc ngắt kết nối)
        if self.gradients is None:
            return np.zeros((x.shape[2], x.shape[3]), dtype=np.float32)

        # Tính trọng số (Weights)
        # Global Average Pooling trên Gradients: Tính độ quan trọng trung bình của từng feature map
        # w shape: [1, Channels, 1, 1]
        w = self.gradients.mean(dim=(2, 3), keepdim=True)
        
        # Tính tổng trọng số các Feature Maps (Weighted Combination)
        # Nhân trọng số (w) với Feature Maps (activations)
        # cam shape: [1, 1, H_feat, W_feat]
        cam = (w * self.activations).sum(dim=1, keepdim=True)
        
        # ReLU (Rectified Linear Unit)
        # Chỉ quan tâm đến các giá trị dương (những vùng ảnh hưởng tích cực đến dự đoán).
        # Những vùng âm (ảnh hưởng tiêu cực) sẽ bị loại bỏ.
        cam = F.relu(cam)
        
        # Resize về kích thước ảnh gốc
        # Dùng nội suy song tuyến tính (bilinear) phóng to heatmap từ 7x7 lên 112x112
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)

        # Chuẩn hóa (Normalization) về [0, 1]
        cam = cam[0, 0].detach().cpu().numpy()
        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()

        return cam

def overlay(img_path, heatmap, size=112, alpha=0.4):
    """
    Trồng lớp heatmap lên ảnh gốc.
    
    Args:
        alpha: Độ trong suốt (0.4 là 40% heatmap, 60% ảnh gốc)
    """
    # Đọc ảnh gốc
    img = cv2.imread(img_path)
    if img is None: return np.zeros((size, size, 3), dtype=np.uint8)
    
    # Resize ảnh gốc và heatmap về cùng kích thước
    img = cv2.resize(img, (size, size))
    heatmap = cv2.resize(heatmap, (size, size))
    
    # Tạo màu cho Heatmap
    # Chuyển từ [0, 1] sang [0, 255]
    heatmap = np.uint8(255 * heatmap)
    # Áp dụng bảng màu JET (Xanh dương = Thấp, Đỏ = Cao)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Trộn ảnh (Blending)
    # out = heatmap * alpha + img * (1 - alpha) + gamma
    out = cv2.addWeighted(heatmap, alpha, img, 1 - alpha, 0)
    
    # Chuyển BGR (OpenCV) sang RGB (Matplotlib)
    return out[:, :, ::-1]

In [ ]:
IDX_TO_GENDER = {0: "Male", 1: "Female"}

IDX_TO_RACE = {
    0: "White", 1: "Black", 2: "Latino", 
    3: "Asian", 4: "Indian"
}

IDX_TO_AGE = {
    0: "0-9", 1: "10-19", 2: "20-29", 
    3: "30-39", 4: "40-59", 5: "60+"
}

def visualize_gradcam_with_labels(model, dataset, device, num_samples=3):
    """
    Vẽ Grad-CAM cho 3 nhiệm vụ (Gender, Race, Age) trên cùng một ảnh.
    Hiển thị: Ảnh gốc | Heatmap Gender | Heatmap Race | Heatmap Age
    Màu tiêu đề: Xanh lá (Đúng) / Đỏ (Sai).
    """
    model.eval()
    
    # Target layer: Chọn layer Conv cuối cùng của backbone EfficientNet
    # Nơi chứa các đặc trưng không gian (spatial features).
    target_layer = model.features[-1] 
    
    # Khởi tạo công cụ GradCAM
    grad_cam = MultiTaskGradCAM(model, target_layer)
    grad_cam.enable() # Bật hook theo dõi gradient
    
    # Chọn ngẫu nhiên num_samples ảnh từ tập test
    indices = random.sample(range(len(dataset)), num_samples)
    
    for idx in indices:
        # Lấy dữ liệu ảnh và nhãn
        img_tensor, label_dict = dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device) # Thêm batch dim [1, C, H, W]
        
        # Lấy đường dẫn ảnh gốc từ DataFrame
        img_path = dataset.df.iloc[idx]["file"]
        
        # PREDICT (DỰ ĐOÁN)
        output = model(input_tensor)
        
        # Gender
        pred_gender_idx = output["gender"].argmax(dim=1).item()
        true_gender_idx = label_dict["gender"].item()
        
        # Race
        pred_race_idx = output["race"].argmax(dim=1).item()
        true_race_idx = label_dict["race"].item()
        
        # Age (Ordinal Regression Decoding)
        # Logic: Sigmoid -> Đếm số lượng output > 0.5
        age_logits = output["age"]
        age_probs = torch.sigmoid(age_logits)
        pred_age_idx = int((age_probs > 0.5).sum(dim=1).item())
        true_age_idx = label_dict["age"].item()
        
        # GENERATE HEATMAPS
        
        # Gender CAM: Soi vào class model dự đoán
        cam_gender = grad_cam.generate(input_tensor, task="gender", class_idx=pred_gender_idx)
        
        # Race CAM
        cam_race = grad_cam.generate(input_tensor, task="race", class_idx=pred_race_idx)
        
        # Age CAM
        # Ordinal Logic: Model có 5 đầu ra (ngưỡng >0, >1, >2, >3, >4).
        # model đoán tuổi là 2 (tức là >1), nên soi vào ngưỡng số 1.
        # target_logit = pred_age - 1 (vì output 0 tương ứng với ngưỡng >0)
        target_logit = max(0, pred_age_idx - 1)
        if target_logit >= 5: target_logit = 4 # Clip max index
        cam_age = grad_cam.generate(input_tensor, task="age", class_idx=target_logit)
        
        # DISPLAY SETUP
        # Trồng lớp heatmap lên ảnh
        vis_gender = overlay(img_path, cam_gender, size=112)
        vis_race   = overlay(img_path, cam_race, size=112)
        vis_age    = overlay(img_path, cam_age, size=112)
        
        # Đọc ảnh gốc bằng OpenCV hiển thị cột đầu tiên
        orig_img = cv2.imread(img_path)
        if orig_img is None: continue
        orig_img = cv2.resize(orig_img, (112, 112))[:, :, ::-1] # BGR to RGB

        # PLOTTING
        fig, axs = plt.subplots(1, 4, figsize=(18, 5))
        
        # Cột 1: Ảnh gốc + Thông tin nhãn thật
        axs[0].imshow(orig_img)
        true_text = (f"TRUE:\n"
                     f"G: {IDX_TO_GENDER.get(true_gender_idx)}\n"
                     f"R: {IDX_TO_RACE.get(true_race_idx)}\n"
                     f"A: {IDX_TO_AGE.get(true_age_idx)}")
        # Vẽ khung text trắng
        axs[0].text(0, -5, true_text, fontsize=10, fontweight='bold', 
                    bbox=dict(facecolor='white', alpha=0.8))
        axs[0].set_title("Original Image")
        axs[0].axis("off")
        
        # Hỗ trợ tô màu tiêu đề (Xanh=Đúng, Đỏ=Sai)
        def set_title_color(ax, pred, true, label_map, task_name):
            pred_name = label_map.get(pred, "Unknown")
            color = 'green' if pred == true else 'red'
            ax.set_title(f"{task_name}: {pred_name}", color=color, fontweight='bold', fontsize=12)

        # Cột 2: Gender Heatmap
        axs[1].imshow(vis_gender)
        set_title_color(axs[1], pred_gender_idx, true_gender_idx, IDX_TO_GENDER, "Pred Gender")
        axs[1].axis("off")
        
        # Cột 3: Race Heatmap
        axs[2].imshow(vis_race)
        set_title_color(axs[2], pred_race_idx, true_race_idx, IDX_TO_RACE, "Pred Race")
        axs[2].axis("off")

        # Cột 4: Age Heatmap
        axs[3].imshow(vis_age)
        set_title_color(axs[3], pred_age_idx, true_age_idx, IDX_TO_AGE, "Pred Age")
        axs[3].axis("off")
        
        plt.tight_layout()
        plt.show()
    
    # Giải phóng bộ nhớ
    grad_cam.disable()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiTaskEfficientNetB0().to(device)

# Đường dẫn file checkpoint
ckpt_path = "/kaggle/working/checkpoints/best_model.pt" 

if os.path.exists(ckpt_path):
    print(f"Loading weights from {ckpt_path}...")
    checkpoint = torch.load(ckpt_path, map_location=device)
    
    state_dict = checkpoint["model_state_dict"]

    # Xử lý lỗi 'module.' nếu model được train bằng DataParallel
    new_state_dict = OrderedDict(
        (k.replace("module.", ""), v) for k, v in state_dict.items()
    )

    # Load weights vào model
    model.load_state_dict(new_state_dict, strict=False) 
    model.eval()
        
# Visualization (Grad-CAM)
visualize_gradcam_with_labels(model, test_ds, device, num_samples=5)

# Ma trận nhầm lẫn (Confusion Matrix)

# Gender
y_true, y_pred = get_all_predictions_multitask(
    model, test_loader, device, task="gender"
)
# GENDER_CLASSES = ["Male", "Female"]
plot_confusion_matrix(y_true, y_pred, GENDER_CLASSES)

# Race
y_true, y_pred = get_all_predictions_multitask(
    model, test_loader, device, task="race"
)
# RACE_CLASSES = ["White", "Black", "Latino", "Asian", "Indian"]
plot_confusion_matrix(y_true, y_pred, RACE_CLASSES)

# Age
y_true, y_pred = get_all_predictions_multitask(
    model, test_loader, device, task="age"
)
# AGE_CLASSES = ["0-9", "10-19", "20-29", "30-39", "40-59", "60+"]
plot_confusion_matrix(y_true, y_pred, AGE_CLASSES)

In [ ]:
CKPT_PATH = "/kaggle/working/checkpoints/best_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Map label (Số) → Tên lớp (String)
gender_map_inv = {0: "Male", 1: "Female"}

race_map_inv = {
    0: "White", 1: "Black", 2: "Latino",
    3: "Asian", 4: "Indian"
}

age_map_inv = {
    0: "0-9", 1: "10-19", 2: "20-29", 3: "30-39",
    4: "40-59", 5: "60+"
}

def process_image(img_path):
    """
    Đọc ảnh và chuyển thành Tensor.
    """
    try:
        img = Image.open(img_path).convert("RGB")
        return val_tf(img).unsqueeze(0) # [1, C, H, W]
    except Exception as e:
        print(f"Lỗi khi đọc ảnh: {e}")
        return None

def visualize_result(img_path, result):
    """
    Hiển thị ảnh gốc và kết quả dự đoán.
    """
    if not os.path.exists(img_path):
        return

    # Mở ảnh gốc hiển thị
    img = Image.open(img_path).convert("RGB")

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off') # Tắt trục tọa độ

    # Tạo tiêu đề
    title_text = (f"Gender: {result['gender']} | "
                  f"Age: {result['age']}\n"
                  f"Race: {result['race']}")
    
    plt.title(title_text, fontsize=12, color='darkblue', fontweight='bold')
    plt.show()

if __name__ == "__main__":
    # Khởi tạo model
    model = MultiTaskEfficientNetB0()
    
    # Load checkpoint
    if os.path.exists(CKPT_PATH):
        try:
            ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
            model.load_state_dict(ckpt["model_state_dict"], strict=False)
            model.to(DEVICE)

            img_path = "/kaggle/input/fairface/FairFace/val/1000.jpg"
            
            if os.path.exists(img_path):
                # Xử lý ảnh
                img_tensor = process_image(img_path)
                img_tensor = img_tensor.to(DEVICE)
                
                if img_tensor is not None:
                    # Dự đoán
                    raw_result = model.predict(img_tensor)
                    
                    final_result = {
                        "gender": gender_map_inv[raw_result["gender"].item()],
                        "race":   race_map_inv[raw_result["race"].item()],
                        "age":    age_map_inv[int(raw_result["age"].item())]
                    }

                    # In kết quả
                    print(f"Raw Tensor: {raw_result}")
                    print(f"Decoded:    {final_result}")
                    
                    # Hiển thị ảnh
                    visualize_result(img_path, final_result)
            else:
                print(f"Không tìm thấy ảnh tại: {img_path}")
                
        except Exception as e:
            print(f"Lỗi khi chạy model: {e}")
    else:
        print(f"Không tìm thấy checkpoint: {CKPT_PATH}")

# Dowload best_model.zip

In [ ]:
def download_file(path, download_file_name):
    os.chdir('/kaggle/working/')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return
    display(FileLink(f'{download_file_name}.zip'))
    
download_file('/kaggle/working/checkpoints/best_model.pt', 'gra_model') 